In [0]:
# ============================================================
# AeroPulse — Maintenance Data Quality
# ============================================================
#
# Purpose
# -------
# Validate Maintenance Bronze records using the reusable
# AeroPulse Data Quality framework.
#
# Source
# ------
# workspace.aeropulse_dev.bronze_maintenance
#
# Data Quality Actions
# --------------------
# VALID   -> eligible for downstream Silver processing
# INVALID -> written to centralized DQ quarantine
#
# Quarantine Table
# ----------------
# workspace.aeropulse_dev.dq_quarantine
#
# Important Design Principle
# --------------------------
# Bronze data is NOT modified or deleted by Data Quality.
#
# Bronze preserves what was received from the source.
#
# Data Quality determines whether records are trusted for
# downstream processing.
# ============================================================

In [0]:
# ============================================================
# Environment Configuration
# ============================================================

ENVIRONMENT = "dev"

CATALOG = "workspace"
SCHEMA = f"aeropulse_{ENVIRONMENT}"

BRONZE_TABLE = (
    f"{CATALOG}.{SCHEMA}.bronze_maintenance"
)

DQ_QUARANTINE_TABLE = (
    f"{CATALOG}.{SCHEMA}.dq_quarantine"
)

SOURCE_SYSTEM = "maintenance_app"
SOURCE_ENTITY = "maintenance"

print(f"Bronze table      : {BRONZE_TABLE}")
print(f"Quarantine table  : {DQ_QUARANTINE_TABLE}")
print(f"Source system     : {SOURCE_SYSTEM}")
print(f"Source entity     : {SOURCE_ENTITY}")

In [0]:
# ============================================================
# Load Reusable Data Quality Framework
# ============================================================
import sys
sys.path.append("workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/data_quality/data_quality_framework")

In [0]:
#============================================================
# Load Reusable Quarantine Framework
# ============================================================
import sys
sys.path.append("workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/data_quality/quarantine")

In [0]:
# ============================================================
# Read Maintenance Bronze
# ============================================================

maintenance_bronze_df = spark.table(
    BRONZE_TABLE
)

print(
    "Maintenance Bronze record count:",
    maintenance_bronze_df.count()
)

In [0]:
display(
    maintenance_bronze_df.limit(20)
)

In [0]:
# ============================================================
# Maintenance Data Quality Rules
# ============================================================

import sys
sys.path.append("workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/data_quality/data_quality_framework")

maintenance_event_id_rule = DataQualityRule(
    rule_name="maintenance_event_id_not_null",
    description="Maintenance event ID must not be NULL",
    check_function=null_check("maintenance_event_id"),
)

aircraft_id_rule = DataQualityRule(
    rule_name="aircraft_id_not_null",
    description="Aircraft ID must not be NULL",
    check_function=null_check("aircraft_id"),
)

engine_id_rule = DataQualityRule(
    rule_name="engine_id_not_null",
    description="Engine ID must not be NULL",
    check_function=null_check("engine_id"),
)

maintenance_cost_rule = DataQualityRule(
    rule_name="maintenance_cost_non_negative",
    description="Maintenance cost must not be negative",
    check_function=non_negative_check(
        "maintenance_cost_usd"
    ),
)

maintenance_dq_rules = [
    maintenance_event_id_rule,
    aircraft_id_rule,
    engine_id_rule,
    maintenance_cost_rule,
]

print(
    f"Configured {len(maintenance_dq_rules)} "
    f"Maintenance Data Quality rules."
)

In [0]:
# ============================================================
# Generate Pipeline Run ID
# ============================================================

import uuid

pipeline_run_id = str(uuid.uuid4())

print(
    f"Pipeline run ID: {pipeline_run_id}"
)

In [0]:
# ============================================================
# Execute Maintenance Data Quality
# ============================================================

import sys
sys.path.append("/Workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/data_quality")
from quarantine import validate_and_quarantine

dq_results = []

for rule in maintenance_dq_rules:

    result = validate_and_quarantine(
        df=maintenance_bronze_df,
        rule=rule,
        pipeline_run_id=pipeline_run_id,
        source_system=SOURCE_SYSTEM,
        source_entity=SOURCE_ENTITY,
        quarantine_table=DQ_QUARANTINE_TABLE,
    )

    dq_results.append(result)

    print(
        f"Rule: {result['rule_name']} | "
        f"Invalid: {result['invalid_records']} | "
        f"Quarantined: {result['quarantined']}"
    )

In [0]:
# ============================================================
# DQ Execution Summary
# ============================================================

dq_results_df = spark.createDataFrame(
    dq_results
)

display(dq_results_df)

In [0]:
# ============================================================
# Step 8.20.1 - Create Controlled Invalid Maintenance Records
# ============================================================

from datetime import datetime, timezone

invalid_test_timestamp = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S_%f")
)

invalid_test_delivery_path = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing/"
    f"{SOURCE_SYSTEM}/{SOURCE_ENTITY}/"
    f"dq_test_{invalid_test_timestamp}"
)

invalid_test_df = spark.createDataFrame(
    [
        (
            "ME_DQ_0001",
            "AC_DQ_0001",
            "EN_DQ_0001",
            "2026-09-21T10:00:00Z",
            15000.00,
            "Bengaluru",
            "completed",
            "inspection",
            "low",
            "TECH_DQ_001",
        ),
        (
            "ME_DQ_0002",
            None,
            "EN_DQ_0002",
            "2026-09-21T10:05:00Z",
            18000.00,
            "Bengaluru",
            "completed",
            "repair",
            "medium",
            "TECH_DQ_002",
        ),
        (
            "ME_DQ_0003",
            "AC_DQ_0003",
            None,
            "2026-09-21T10:10:00Z",
            22000.00,
            "London",
            "completed",
            "inspection",
            "low",
            "TECH_DQ_003",
        ),
        (
            "ME_DQ_0004",
            "AC_DQ_0004",
            "EN_DQ_0004",
            "2026-09-21T10:15:00Z",
            -5000.00,
            "Dubai",
            "completed",
            "repair",
            "high",
            "TECH_DQ_004",
        ),
    ],
    [
        "maintenance_event_id",
        "aircraft_id",
        "engine_id",
        "event_timestamp",
        "maintenance_cost_usd",
        "maintenance_location",
        "maintenance_status",
        "maintenance_type",
        "severity",
        "technician_id",
    ],
)

print(
    f"Test delivery path: {invalid_test_delivery_path}"
)

print(
    f"Test records: {invalid_test_df.count()}"
)

display(invalid_test_df)

In [0]:
# ============================================================
# Step 8.20.2 - Land Invalid Test Delivery
# ============================================================

(
    invalid_test_df.write
    .mode("error")
    .json(invalid_test_delivery_path)
)

print(
    f"Invalid test delivery landed successfully:"
)
print(
    invalid_test_delivery_path
)

In [0]:
display(
    dbutils.fs.ls(
        invalid_test_delivery_path
    )
)

In [0]:
# ============================================================
# Step 8.20.3 - Ingest DQ Test Delivery into Bronze
# ============================================================

SOURCE_PATH = invalid_test_delivery_path

CHECKPOINT_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/"
    f"raw_landing/_checkpoints/"
    f"{SOURCE_SYSTEM}/{SOURCE_ENTITY}/dq_test"
)

maintenance_test_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        CHECKPOINT_PATH
    )
    .load(SOURCE_PATH)
)

In [0]:
maintenance_test_bronze_df = (
    maintenance_test_stream_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY)
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
)

In [0]:
dq_test_bronze_query = (
    maintenance_test_bronze_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        BRONZE_TABLE
    )
)

dq_test_bronze_query.awaitTermination()

In [0]:
# ============================================================
# Step 8.20.4 - Verify Test Records in Bronze
# ============================================================

display(
    spark.sql(f"""
        SELECT
            maintenance_event_id,
            aircraft_id,
            engine_id,
            maintenance_cost_usd,
            _source_file_path,
            _ingestion_timestamp
        FROM {BRONZE_TABLE}
        WHERE maintenance_event_id LIKE 'ME_DQ_%'
        ORDER BY maintenance_event_id
    """)
)

In [0]:
# ============================================================
# Step 8.21.1 - Load DQ Test Records from Bronze
# ============================================================

dq_test_df = (
    spark.table(BRONZE_TABLE)
    .filter(
        F.col("maintenance_event_id").like("ME_DQ_%")
    )
)

print(
    f"DQ test records found in Bronze: "
    f"{dq_test_df.count()}"
)

display(dq_test_df)

In [0]:
# ============================================================
# Step 8.21.2 - DQ Test Pipeline Run ID
# ============================================================

import uuid

dq_test_pipeline_run_id = str(uuid.uuid4())

print(
    f"DQ test pipeline run ID: "
    f"{dq_test_pipeline_run_id}"
)

In [0]:
# ============================================================
# Step 8.21.3 - Execute DQ Against Test Records
# ============================================================
dq_test_results = []

for rule in maintenance_dq_rules:

    result = validate_and_quarantine(
        df=dq_test_df,
        rule=rule,
        pipeline_run_id=dq_test_pipeline_run_id,
        source_system=SOURCE_SYSTEM,
        source_entity=SOURCE_ENTITY,
        quarantine_table=DQ_QUARANTINE_TABLE,
    )

    dq_test_results.append(result)

    print(
        f"Rule: {result['rule_name']} | "
        f"Invalid: {result['invalid_records']} | "
        f"Quarantined: {result['quarantined']}"
    )

In [0]:
# ============================================================
# Step 8.21.4 - DQ Test Summary
# ============================================================

display(
    spark.createDataFrame(dq_test_results)
)

In [0]:
# ============================================================
# Step 8.21.5 - Verify Quarantine Records
# ============================================================

display(
    spark.sql(f"""
        SELECT
            quarantine_event_id,
            pipeline_run_id,
            source_system,
            source_entity,
            rule_name,
            failure_reason,
            source_file_path,
            quarantine_timestamp,
            record_json
        FROM {DQ_QUARANTINE_TABLE}
        WHERE pipeline_run_id = '{dq_test_pipeline_run_id}'
        ORDER BY rule_name
    """)
)

In [0]:
# ============================================================
# Step 8.21.6 - Prove Bronze Was Not Modified
# ============================================================

display(
    spark.sql(f"""
        SELECT
            maintenance_event_id,
            aircraft_id,
            engine_id,
            maintenance_cost_usd
        FROM {BRONZE_TABLE}
        WHERE maintenance_event_id LIKE 'ME_DQ_%'
        ORDER BY maintenance_event_id
    """)
)

In [0]:
# ============================================================
# Step 8.21.7 - Count Quarantined Events
# ============================================================

quarantine_count = spark.sql(f"""
    SELECT COUNT(*) AS quarantine_count
    FROM {DQ_QUARANTINE_TABLE}
    WHERE pipeline_run_id = '{dq_test_pipeline_run_id}'
""").collect()[0]["quarantine_count"]

print(
    f"Quarantine records for this DQ run: "
    f"{quarantine_count}"
)

In [0]:
# ============================================================
# Step 8.22.1 - Capture Current Quarantine Count
# ============================================================

before_rerun_count = spark.sql(f"""
    SELECT COUNT(*) AS quarantine_count
    FROM {DQ_QUARANTINE_TABLE}
    WHERE pipeline_run_id = '{dq_test_pipeline_run_id}'
""").collect()[0]["quarantine_count"]

print(
    f"Quarantine records before rerun: {before_rerun_count}"
)

In [0]:
# ============================================================
# Step 8.22.2 - Rerun the Same DQ Execution
# ============================================================

dq_rerun_results = []

for rule in maintenance_dq_rules:

    result = validate_and_quarantine(
        df=dq_test_df,
        rule=rule,
        pipeline_run_id=dq_test_pipeline_run_id,
        source_system=SOURCE_SYSTEM,
        source_entity=SOURCE_ENTITY,
        quarantine_table=DQ_QUARANTINE_TABLE,
    )

    dq_rerun_results.append(result)

    print(
        f"Rule: {result['rule_name']} | "
        f"Invalid: {result['invalid_records']} | "
        f"Quarantined: {result['quarantined']}"
    )

In [0]:
# ============================================================
# Step 8.22.3 - Verify Idempotency
# ============================================================

after_rerun_count = spark.sql(f"""
    SELECT COUNT(*) AS quarantine_count
    FROM {DQ_QUARANTINE_TABLE}
    WHERE pipeline_run_id = '{dq_test_pipeline_run_id}'
""").collect()[0]["quarantine_count"]

print(
    f"Quarantine records after rerun: {after_rerun_count}"
)

print(
    f"Count unchanged: "
    f"{before_rerun_count == after_rerun_count}"
)

In [0]:
# ============================================================
# Step 8.22.4 - Verify Unique Quarantine Event IDs
# ============================================================

display(
    spark.sql(f"""
        SELECT
            quarantine_event_id,
            rule_name,
            pipeline_run_id,
            COUNT(*) AS event_count
        FROM {DQ_QUARANTINE_TABLE}
        WHERE pipeline_run_id = '{dq_test_pipeline_run_id}'
        GROUP BY
            quarantine_event_id,
            rule_name,
            pipeline_run_id
        ORDER BY rule_name
    """)
)

In [0]:
# ============================================================
# Step 8.22.5 - Detect Duplicate Quarantine Events
# ============================================================

duplicate_events_df = spark.sql(f"""
    SELECT
        quarantine_event_id,
        COUNT(*) AS event_count
    FROM {DQ_QUARANTINE_TABLE}
    WHERE pipeline_run_id = '{dq_test_pipeline_run_id}'
    GROUP BY quarantine_event_id
    HAVING COUNT(*) > 1
""")

duplicate_count = duplicate_events_df.count()

print(
    f"Duplicate quarantine events: {duplicate_count}"
)

In [0]:
# ============================================================
# Step 8.22.6 - Idempotency Assertion
# ============================================================

assert before_rerun_count == after_rerun_count, (
    "Idempotency validation failed: "
    "quarantine count changed after rerun."
)

assert duplicate_count == 0, (
    "Idempotency validation failed: "
    "duplicate quarantine events detected."
)

print(
    "SUCCESS: Quarantine processing is idempotent."
)